# SciPy Optimization Bridge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mitchell-Mirano/sorix/blob/develop/docs/learn/optimizers/07-ScipyBridge.ipynb)
[![Open in GitHub](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/Mitchell-Mirano/sorix/blob/develop/docs/learn/optimizers/07-ScipyBridge.ipynb)
[![Open in Docs](https://img.shields.io/badge/Open%20in-Docs-blue?logo=readthedocs)](http://127.0.0.1:8000/sorix/learn/optimizers/07-ScipyBridge)

In scientific machine learning and engineering applications, we frequently encounter **inverse design** and **constrained optimization** problems. For example, in concrete mix optimization, we want to find the precise proportions of cement, water, and aggregates that yield a target compressive strength while satisfying physical bounds (e.g., ingredient ratios between 0 and 1).

While `sorix` provides native neural network optimizers (like Adam and SGD), mathematical libraries like **SciPy** feature highly optimized, classical optimization algorithms that support complex bounds and non-linear constraints (such as `L-BFGS-B` and `SLSQP`).

The `ScipyBridge` utility bridges the gap between `sorix` autograd and SciPy's solvers, combining **exact analytical gradients** with **classical constrained optimization**.

## 1. Why use `ScipyBridge`?

When using SciPy's optimizers with machine learning models, developers typically face two major challenges:

1. **Numerical Finite Differences are Slow and Unstable**: By default, SciPy approximates gradients using finite differences (evaluating the objective function $2D$ times for $D$ variables). This is computationally expensive and numerically unstable (highly sensitive to step size $h$). `ScipyBridge` solves this by calculating exact analytical gradients via `sorix` backpropagation.
2. **CPU-GPU Memory Boundary**: SciPy is a CPU-bound library that expects CPU NumPy arrays. If your `sorix` model runs on GPU (via CuPy/CUDA), `ScipyBridge` automatically handles copying candidates onto the GPU, evaluating the model, and returning gradients back to CPU NumPy arrays seamlessly.

In [ ]:
# Uncomment to install sorix
#!pip install 'sorix @ git+https://github.com/Mitchell-Mirano/sorix.git@develop'

## 2. Example 1: Basic Function Minimization

Let's start with a simple optimization problem: minimizing a 2D quadratic bowl function:
$$f(x, y) = (x - 2)^2 + (y - 3)^2 + 5$$

We will initialize our variables at `[0.0, 0.0]` and use SciPy's `L-BFGS-B` algorithm to find the optimum at `[2.0, 3.0]`.

In [ ]:
import numpy as np
from sorix import tensor
from sorix.optim import ScipyBridge
import scipy.optimize

# 1. Define the parameters to optimize (requires_grad must be True)
params = tensor([0.0, 0.0], requires_grad=True)

# 2. Define the objective function (must return a scalar loss Tensor)
def loss_fn():
    return (params[0] - 2.0)**2 + (params[1] - 3.0)**2 + 5.0

# 3. Wrap with ScipyBridge
bridge = ScipyBridge(params, loss_fn)

# 4. Optimize using SciPy L-BFGS-B
res = scipy.optimize.minimize(
    bridge.objective,
    bridge.get_x(),
    jac=True,  # Tell SciPy that the objective function returns both (value, gradient)
    method='L-BFGS-B'
)

print("Optimization Successful:", res.success)
print("Optimal x, y:", res.x)
print("Minimum loss value:", res.fun)

## 3. Example 2: Inverse Design (Input Space Optimization) with Physical Bounds

In physical engineering problems (like concrete mix design), we train a neural network to predict a physical property (e.g., strength) from input variables (e.g., cement, water, aggregate fractions).

Once the network is trained, we perform **inverse design**: optimizing the inputs to hit a target strength. Here, we must also enforce physical bounds (e.g., ingredient ratios must lie in the range $[0, 1]$).

Let's build a simple trained MLP, freeze its weights, and run bounds-constrained optimization on the input features.

In [ ]:
from sorix.nn import Linear, Sequential

# 1. Setup a dummy 'trained' model (3 input ingredients, 1 output strength)
model = Sequential(
    Linear(3, 8),
    Linear(8, 1)
)

# Freeze model weights: during inverse design, we ONLY optimize the input variables
for p in model.parameters():
    p.requires_grad = False

# 2. Define the input decision variables and set initial guess
x_input = tensor([0.5, 0.5, 0.5], requires_grad=True)

# 3. Define target strength we want to achieve
target_strength = 12.0

# 4. Define the objective function
def objective_fn():
    pred = model(x_input)
    # Penalize deviation from target strength
    prediction_loss = (pred - target_strength)**2
    # Penalize excessive cement (ingredient 0) usage to keep the mix cost-efficient
    cement_cost = 0.5 * (x_input[0] ** 2)
    return prediction_loss + cement_cost

# 5. Define physical bounds for the ingredients: [0.0, 1.0] for each ingredient
bounds = [
    (0.0, 1.0),  # Cement bounds
    (0.1, 0.9),  # Water bounds
    (0.0, 1.0)   # Aggregate bounds
]

# 6. Initialize bridge
bridge = ScipyBridge(x_input, objective_fn)

# 7. Run bounded optimization
res = scipy.optimize.minimize(
    bridge.objective,
    bridge.get_x(),
    jac=True,
    bounds=bounds,
    method='L-BFGS-B'
)

print("Optimization Successful:", res.success)
print("Optimal inputs:", res.x)
print("Predicted strength at optimum:", model(tensor(res.x)).item())

## 4. Key Performance Guidelines

When using `ScipyBridge`, keep these important design tips in mind:

1. **Call `model.eval()`**: Always set your neural network to evaluation mode before running optimization. This disables dropout noise and freezes batch normalization running statistics, ensuring a smooth, noise-free optimization surface.
2. **Freeze Weights**: If you are optimizing the input features, remember to iterate over `model.parameters()` and set `requires_grad = False` to prevent computing gradients for network weights, saving memory and runtime.
3. **Memory Management**: `ScipyBridge` automatically calls `backward(retain_graph=False)` on the loss, freeing autograd graph memory at each step. This allows optimization runs to execute for hundreds of steps with constant memory usage.